# L2: Unsteady Convection-Reaction

Convection is the dominant transport mechanism in tubular reactors and column flows. This notebook covers the **method of lines** (MOL) discretisation of the 1D convection-reaction equation on a cell-centred grid, the stability requirement (CFL/Courant condition), and the improvement in accuracy achieved by TVD (total-variation-diminishing) limiters compared with first-order upwind (FOU).

## Governing equation

$$\frac{\partial c}{\partial t} + v \frac{\partial c}{\partial z} = R(c)$$

Discretised on $N$ cells of width $\Delta z = L/N$, the FOU convective flux at face $i+\tfrac{1}{2}$:

$$F_{i+1/2} = v\, c_i \quad (v > 0)$$

Von Neumann stability for explicit Euler + FOU:

$$\text{Cr} = \frac{v\,\Delta t}{\Delta z} \leq 1$$

TVD flux with limiter $\phi(r)$ (minmod, van Leer, …):

$$F_{i+1/2}^{\text{TVD}} = v\left[c_i + \tfrac{1}{2}\phi(r_i)(c_{i+1}-c_i)\right]$$

where $r_i = (c_i - c_{i-1})/(c_{i+1} - c_i)$ is the slope ratio.

## PyMRM building blocks

| Function | Role |
|---|---|
| `construct_convflux_upwind(v, N, dz)` | Sparse matrix for first-order upwind fluxes |
| `interp_cntr_to_stagg_tvd(c, v, limiter)` | TVD-corrected face concentrations |
| `construct_div(N, dz)` | Divergence operator: maps face fluxes → cell residuals |
| `minmod`, `vanleer` | Limiter functions $\phi(r)$ |

## Example 1 — Step input: FOU vs TVD

Inject a sharp step (Heaviside) at $z=0$ and observe numerical diffusion (FOU) versus oscillation-free sharpening (TVD minmod).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pymrm import construct_convflux_upwind, interp_cntr_to_stagg_tvd, construct_div

# Grid
L, N = 1.0, 100
dz = L / N
z = (np.arange(N) + 0.5) * dz
v = 1.0        # [m/s]
dt = 0.8 * dz / v   # Courant number = 0.8
t_end = 0.5

# Initial condition: step at z = 0.2
c0 = (z >= 0.2).astype(float)
c_fou = c0.copy()
c_tvd = c0.copy()

# Operators
Div = construct_div(N, dz)
A_fou = construct_convflux_upwind(v * np.ones(N+1), N, dz)

t = 0.0
while t < t_end - 1e-12:
    dt_step = min(dt, t_end - t)
    # FOU
    c_fou -= dt_step * (Div @ A_fou @ np.pad(c_fou, 1)[:N+1+1][:N+1]  # boundary flux
                        )
    # TVD (deferred correction on top of FOU)
    c_face = interp_cntr_to_stagg_tvd(c_tvd, v * np.ones(N+1), 'minmod')
    c_tvd -= dt_step * (Div @ (v * c_face))
    t += dt_step

exact = (z >= 0.2 + v * t_end).astype(float)
plt.figure(figsize=(6, 3.5))
plt.plot(z, exact, 'k-', lw=2, label='Exact')
plt.plot(z, c_fou, '--', label='FOU')
plt.plot(z, c_tvd, '-', label='TVD minmod')
plt.xlabel('z (m)'); plt.ylabel('c')
plt.title(f'Convection at t = {t_end:.2f} s,  N = {N},  Cr = {v*dt/dz:.2f}')
plt.legend(); plt.tight_layout(); plt.show()

## Example 2 — Convection-reaction: A → B

First-order irreversible reaction $R = -k c_A$ superimposed on convection.

In [ ]:
k_rxn = 2.0    # [1/s]
c_A = np.zeros(N); c_A[0] = 0.0  # inlet at left
c_B = np.zeros(N)
# Steady-state inlet: c_A_in = 1
c_A_in = 1.0
c_A = np.ones(N)  # reset
c_B = np.zeros(N)

# Integrate to steady state
t = 0.0; t_end2 = 4.0
while t < t_end2:
    dt2 = min(dt, t_end2 - t)
    cA_face = interp_cntr_to_stagg_tvd(c_A, v * np.ones(N+1), 'minmod')
    cA_face[0] = c_A_in   # inlet BC
    cB_face = interp_cntr_to_stagg_tvd(c_B, v * np.ones(N+1), 'minmod')
    cB_face[0] = 0.0
    c_A += dt2 * (-Div @ (v * cA_face) - k_rxn * c_A)
    c_B += dt2 * (-Div @ (v * cB_face) + k_rxn * c_A)
    t += dt2

Da = k_rxn * L / v
c_A_exact = c_A_in * np.exp(-k_rxn * z / v)
plt.figure(figsize=(6, 3.5))
plt.plot(z, c_A_exact, 'k-', label='c_A exact')
plt.plot(z, c_A, '--', label='c_A TVD')
plt.plot(z, c_B, '-', label='c_B TVD')
plt.xlabel('z (m)'); plt.ylabel('c (mol/m³)')
plt.title(f'Convection-reaction, Da = {Da:.1f}')
plt.legend(); plt.tight_layout(); plt.show()

## Summary

- **Method of lines**: discretise space → system of ODEs in time
- **FOU**: $O(\Delta z)$ accuracy, high numerical diffusion, stable for Cr ≤ 1
- **TVD**: $O(\Delta z^2)$ in smooth regions, no new extrema, same CFL constraint
- **Normalised variable diagram (NVD)**: all TVD limiters live in the Sweby region
- Stiffness arises when $k\,\Delta t \gg 1$; use implicit methods or operator splitting